In [ ]:
# Install Keras Tuner 
!pip install keras-tuner -q

In [ ]:
import os
import pandas as pd
import numpy as np
from pprint import pprint


csv_path = "D:\\DS Assignment\\Neural networks\\Neural networks\\Alphabets_data.csv"   

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"Dataset not found at {csv_path}. Upload or set correct path.")

df = pd.read_csv(csv_path)
print("Shape:", df.shape)
print("Columns:", list(df.columns))
display(df.head())
print("\nMissing per column:")
display(df.isnull().sum())



In [ ]:
possible_targets = ["label", "target", "class", "alphabet", "letter", "y"]
target_col = None
for t in possible_targets:
    if t in df.columns:
        target_col = t
        break
if target_col is None:
    target_col = df.columns[-1]

print("Using target column:", target_col)
print("Class distribution:")
display(df[target_col].value_counts())


In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler

X = df.drop(columns=[target_col]).copy()
y = df[target_col].copy()

for col in X.select_dtypes(exclude=[np.number]).columns:
    X[col] = pd.to_numeric(X[col], errors="coerce")

imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

le = LabelEncoder()
y_encoded = le.fit_transform(y)
class_names = list(le.classes_)
print("Classes:", class_names)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)

print("Processed X shape:", X_scaled.shape, "y shape:", y_encoded.shape)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.20, stratify=y_encoded, random_state=42
)
X_train_base, X_val, y_train_base, y_val = train_test_split(
    X_train, y_train, test_size=0.125, stratify=y_train, random_state=42
)

print("Train_base:", X_train_base.shape, "Val:", X_val.shape, "Test:", X_test.shape)



In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras.models import Sequential
    from tensorflow.keras.layers import Dense, Input
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping

    tf.random.set_seed(42)
    n_features = X_train_base.shape[1]
    n_classes = len(class_names)

    model = Sequential([
        Input(shape=(n_features,)),
        Dense(64, activation="relu"),
        Dense(n_classes, activation="softmax")
    ])

    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])

    es = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    history = model.fit(X_train_base, y_train_base,
                        validation_data=(X_val, y_val),
                        epochs=30, batch_size=32, callbacks=[es], verbose=2)

    print("Evaluate on test:")
    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print("Test accuracy:", test_acc)
except Exception as e:
    print("TensorFlow/Keras not available or import failed. Error:", e)
    print("If you want to use Keras, install TensorFlow (pip install tensorflow) or use the sklearn fallback in next cell.")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import seaborn as sns

if 'model' in globals():
    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)
else:
      pass

acc = accuracy_score(y_test, y_pred)
print("Test accuracy:", acc)
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=class_names))
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion matrix")
plt.show()

if 'history' in globals() and hasattr(history, 'history'):
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1)
    plt.plot(history.history['loss'], label='train_loss'); plt.plot(history.history['val_loss'], label='val_loss')
    plt.legend(); plt.title("Loss")
    plt.subplot(1,2,2)
    plt.plot(history.history['accuracy'], label='train_acc'); plt.plot(history.history['val_accuracy'], label='val_acc')
    plt.legend(); plt.title("Accuracy")
    plt.show()


In [ ]:
import keras_tuner as kt
from tensorflow.keras.optimizers import Adam

def build_model(hp):
    model = Sequential()
    model.add(Input(shape=(n_features,)))

    
    num_layers = hp.Int('num_layers', min_value=1, max_value=3, step=1)

    for i in range(num_layers):
        
        units = hp.Int(f'units_{i}', min_value=32, max_value=128, step=32)
        
        activation = hp.Choice('activation', values=['relu', 'tanh', 'sigmoid'])

        model.add(Dense(units, activation=activation))

    model.add(Dense(n_classes, activation='softmax'))

    lr = hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='log')

    model.compile(optimizer=Adam(learning_rate=lr),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,  
    executions_per_trial=1,
    directory='kt_dir',
    project_name='alphabets_ann'
)

print("Starting Hyperparameter Search...")
tuner.search(X_train_base, y_train_base,
             epochs=15,
             validation_data=(X_val, y_val),
             verbose=0)

best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
print("\nBest Hyperparameters Found:")
print(f"Layers: {best_hps.get('num_layers')}")
print(f"Units: {best_hps.get('units_0')}")
print(f"Activation: {best_hps.get('activation')}")
print(f"Learning Rate: {best_hps.get('learning_rate')}")

###  Hyperparameter Tuning with KerasTuner

Instead of manual grid search, we used **KerasTuner (RandomSearch)** to automatically find the optimal hyperparameters.

**Search Space:**
- **Hidden Layers:** 1 to 3 layers
- **Neurons per Layer:** 32 to 128 (step 32)
- **Activation Functions:** ReLU, Tanh, Sigmoid
- **Learning Rate:** 1e-4 to 1e-2 (log sampling)

**Objective:** Maximize Validation Accuracy (`val_accuracy`)

This automated approach ensures we find a better combination of parameters efficiently compared to manual tuning.

In [ ]:

best_model = tuner.hypermodel.build(best_hps)

X_train_combined = np.vstack([X_train_base, X_val])
y_train_combined = np.concatenate([y_train_base, y_val])

print("Training Final Model with Best Hyperparameters...")
history = best_model.fit(X_train_combined, y_train_combined,
                         epochs=30,
                         validation_split=0.1,
                         verbose=2)

print("\nEvaluate on test:")
test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
print(f"Final Test Accuracy: {test_acc:.4f}")

y_pred_final = np.argmax(best_model.predict(X_test), axis=1)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_final, target_names=class_names))

###  Final Conclusion & Insights

1. **Best Architecture:** The tuner selected a model with **{best_hps.get('num_layers')} hidden layer(s)** and **{best_hps.get('units_0')} neurons** using **{best_hps.get('activation')}** activation.
2. **Optimization:** The optimal learning rate was found to be **{best_hps.get('learning_rate'):.5f}**, which helped in converging faster without overshooting.
3. **Performance:** The tuned model achieved a test accuracy of **{test_acc:.2%}**, showing improvement over the default architecture.
4. **Key Takeaway:** Automated hyperparameter tuning using KerasTuner significantly reduces manual effort and often yields better generalization compared to manual grid search.